# Hatteras Island CASCADE Hindcast
Step through each section, run it, look at the plot, confirm it's right before moving on.
Single configuration only (one `START_YEAR`, one `Hs`) — the sensitivity sweep stays in `HAT_groin_sensitivity_sweep.py`.

Pseudocode only below — move your own working code in from `HAT_hindcast_1984_2024_groinTest.py` section by section. Line numbers referenced are from that file.

## 1. Imports

In [9]:
import os
import sys
from pathlib import Path

# cascade_pipeline and hatteras_site_config live in scripts/, which isn't installed
_here = Path.cwd().resolve()
_repo_root = next((p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()), None)
if _repo_root is None:
    raise RuntimeError(f"CASCADE repo root not found above {_here}")
SCRIPTS_DIR = _repo_root / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# True: sandbox copy with the pre-AST groin hook. False: real package, hook folded in.
USE_SANDBOX_CASCADE = True
if USE_SANDBOX_CASCADE:
    from cascade.cascade_groin import Cascade
else:
    from cascade.cascade import Cascade

from cascade_pipeline.annotations import AnnotationConfig
from cascade_pipeline.coastsat_loess import (
    DEFAULT_LOESS,
    CoastSatDataset,
    LoessConfig,
    build_coastsat_series,
)
from cascade_pipeline.plotting.rate_comparison import (
    plot_annotated_rate_comparison,
    plot_rate_comparison,
)
from cascade_pipeline.plotting.shoreline_gif import (
    GifConfig,
    make_all_shoreline_gifs,
    make_shoreline_gif,
)
from cascade_pipeline.run_info import RunInfo
from cascade_pipeline.shoreline import build_shoreline_matrix, compute_change_rate
import PIL.Image
from IPython.display import display

from hatteras_site_config import HATTERAS_ANNOTATIONS, HATTERAS_DOMAINS

print(f"Imports OK from {SCRIPTS_DIR}")
print(f"USE_SANDBOX_CASCADE = {USE_SANDBOX_CASCADE}")
print(f"HATTERAS_DOMAINS.total_domains = {HATTERAS_DOMAINS.total_domains}")


Imports OK from C:\Users\hanna\PycharmProjects\CASCADE\scripts
USE_SANDBOX_CASCADE = True
HATTERAS_DOMAINS.total_domains = 120


## 2. Fixed dune/topo (period-independent) — QC plot
Same 2009 init surface for both periods, doesn't depend on `START_YEAR`.

Broken into steps so each can be confirmed before moving on:

| Step | What it does | What to check |
| --- | --- | --- |
| 2.1 | Resolve project paths | printed dirs all say `ok` |
| 2.2 | Build the 120 padded file paths | buffer/real/buffer order, 120 each |
| 2.3 | Verify every file exists | no exception |
| 2.4 | Units check vs Barrier3D contract | every check reports 90/90 |
| 2.5 | Browse per-domain extractor figures | dune picks and water trim, domain by domain |

### 2.1 Project paths

Everything derives from `SCRIPTS_DIR` (Section 1), so no absolute paths are
hardcoded. Layout under `data/hatteras_init/1-barrier3d-domains/`:

```
2009-dune-topo/<version>/topography/domain_<gis>_topography_<year>.npy
2009-dune-topo/<version>/dunes/domain_<gis>_dune_<year>.npy
2009-buffer/sample_1_topography.npy      # one profile, reused by every buffer
2009-buffer/sample_1_dune.npy
```

`TOPO_DUNE_VERSION` switches extractor versions without touching the filename
year. `2009_v2` is what the hindcast run script uses.

In [10]:
TOPO_DUNE_INIT_YEAR = "2009"     # year label inside the filenames
TOPO_DUNE_VERSION = "2009_v2"    # extractor version folder

PROJECT_BASE_DIR = SCRIPTS_DIR.parent
HATTERAS_DATA_BASE = PROJECT_BASE_DIR / "data" / "hatteras_init"
OUTPUT_BASE_DIR = PROJECT_BASE_DIR / "comparison" / "raw_runs"
COASTSAT_BASE_DIR = (PROJECT_BASE_DIR / "scripts" / "input_prep"
                     / "5-scr" / "CoastSat")
PARAMETER_FILE = "Hatteras-CASCADE-parameters.yaml"  # resolved by CASCADE

BARRIER3D_DIR = HATTERAS_DATA_BASE / "1-barrier3d-domains"
DUNE_TOPO_DIR = BARRIER3D_DIR / "2009-dune-topo" / TOPO_DUNE_VERSION
BUFFER_DIR = BARRIER3D_DIR / "2009-buffer"

os.chdir(PROJECT_BASE_DIR)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

for _name, _path in [
    ("PROJECT_BASE_DIR", PROJECT_BASE_DIR),
    ("HATTERAS_DATA_BASE", HATTERAS_DATA_BASE),
    ("DUNE_TOPO_DIR", DUNE_TOPO_DIR),
    ("BUFFER_DIR", BUFFER_DIR),
    ("COASTSAT_BASE_DIR", COASTSAT_BASE_DIR),
    ("OUTPUT_BASE_DIR", OUTPUT_BASE_DIR),
]:
    print(f"{_name:<20} {'ok' if _path.exists() else 'MISSING':<8} {_path}")

PROJECT_BASE_DIR     ok       C:\Users\hanna\PycharmProjects\CASCADE
HATTERAS_DATA_BASE   ok       C:\Users\hanna\PycharmProjects\CASCADE\data\hatteras_init
DUNE_TOPO_DIR        ok       C:\Users\hanna\PycharmProjects\CASCADE\data\hatteras_init\1-barrier3d-domains\2009-dune-topo\2009_v2
BUFFER_DIR           ok       C:\Users\hanna\PycharmProjects\CASCADE\data\hatteras_init\1-barrier3d-domains\2009-buffer
COASTSAT_BASE_DIR    ok       C:\Users\hanna\PycharmProjects\CASCADE\scripts\input_prep\5-scr\CoastSat
OUTPUT_BASE_DIR      ok       C:\Users\hanna\PycharmProjects\CASCADE\comparison\raw_runs


### 2.2 Build the padded file lists

CASCADE takes one elevation file and one dune file per padded domain, in
alongshore order: 15 buffers, then real GIS 1–90, then 15 buffers (120 total).
Both buffer ends reuse the same sample profile, so only the real domains vary.

The returned lists are index-aligned with the padded array, so
`HATTERAS_DOMAINS.gis_to_pad(gis_id)` indexes straight into them.

In [11]:
def build_domain_file_paths(geometry, dune_topo_dir, buffer_dir, init_year):
    """Builds one elevation and one dune file path per padded domain.

    Args:
        geometry: DomainGeometry describing the padded domain array.
        dune_topo_dir: Directory holding the topography/ and dunes/ subdirs.
        buffer_dir: Directory holding the buffer sample profiles.
        init_year: Year label embedded in the real-domain filenames.

    Returns:
        An (elevation_paths, dune_paths) tuple of string lists. Each list is
        geometry.total_domains long and index-aligned with the padded array.
    """
    buffer_elevation = str(buffer_dir / "sample_1_topography.npy")
    buffer_dune = str(buffer_dir / "sample_1_dune.npy")

    elevation_paths = [buffer_elevation] * geometry.num_buffer_domains
    dune_paths = [buffer_dune] * geometry.num_buffer_domains

    for gis_id in range(geometry.first_gis_id, geometry.last_gis_id + 1):
        elevation_paths.append(str(
            dune_topo_dir / "topography"
            / f"domain_{gis_id}_topography_{init_year}.npy"))
        dune_paths.append(str(
            dune_topo_dir / "dunes"
            / f"domain_{gis_id}_dune_{init_year}.npy"))

    elevation_paths += [buffer_elevation] * geometry.num_buffer_domains
    dune_paths += [buffer_dune] * geometry.num_buffer_domains

    return elevation_paths, dune_paths


ELEVATION_FILE_PATHS, DUNE_FILE_PATHS = build_domain_file_paths(
    HATTERAS_DOMAINS, DUNE_TOPO_DIR, BUFFER_DIR, TOPO_DUNE_INIT_YEAR)

_first_real = HATTERAS_DOMAINS.start_real_index
_last_real = HATTERAS_DOMAINS.end_real_index - 1

print(f"{len(ELEVATION_FILE_PATHS)} elevation + {len(DUNE_FILE_PATHS)} dune "
      f"paths (expect {HATTERAS_DOMAINS.total_domains} each)\n")
print("Padded array boundaries:")
for _label, _pad in [
    ("pad 0 (buffer)", 0),
    (f"pad {_first_real} (GIS {HATTERAS_DOMAINS.first_gis_id})", _first_real),
    (f"pad {_last_real} (GIS {HATTERAS_DOMAINS.last_gis_id})", _last_real),
    (f"pad {HATTERAS_DOMAINS.total_domains - 1} (buffer)",
     HATTERAS_DOMAINS.total_domains - 1),
]:
    print(f"  {_label:<22} {Path(ELEVATION_FILE_PATHS[_pad]).name}")

120 elevation + 120 dune paths (expect 120 each)

Padded array boundaries:
  pad 0 (buffer)         sample_1_topography.npy
  pad 15 (GIS 1)         domain_1_topography_2009.npy
  pad 104 (GIS 90)       domain_90_topography_2009.npy
  pad 119 (buffer)       sample_1_topography.npy


### 2.3 Verify every file exists

Checks all 240 paths up front. A stale `TOPO_DUNE_VERSION` or a moved data
folder fails here with a count and the first offender, rather than surfacing
as an empty plot or an opaque traceback inside Barrier3D's init.

In [12]:
_expected_files = 2 * HATTERAS_DOMAINS.total_domains
_missing = [path for path in ELEVATION_FILE_PATHS + DUNE_FILE_PATHS
            if not Path(path).exists()]

if _missing:
    raise FileNotFoundError(
        f"{len(_missing)} of {_expected_files} init files missing. Check "
        f"TOPO_DUNE_VERSION ({TOPO_DUNE_VERSION!r}) and DUNE_TOPO_DIR.\n"
        f"  First missing: {_missing[0]}")

print(f"All {_expected_files} init files present.")

All 240 init files present.


### 2.4 Units check against Barrier3D's input contract

Barrier3D's `load_input.py` converts the *scalar* YAML parameters but loads the
elevation and dune `.npy` files **verbatim** — `InteriorDomain` is whatever the
file contains, with no unit conversion applied:

```python
params["MHW"] /= 10.0                                  # m   -> dam
params["BarrierLength"] = int(params["BarrierLength"] / 10.0)   # m -> cells
params["BermEl"] = params["BermEl"] / 10.0 - params["MHW"]      # m -> dam above MHW
params["InteriorDomain"] = load_elevation(...)         # <- no conversion
```

So the arrays on disk must *already* be in decameters relative to MHW. A file
written in metres would run without error and silently model an island 10x too
tall. This step verifies that, plus the shape contract `load_input.py` relies
on, across all 90 real domains — deliberately on the **raw** arrays, before any
`DAM_TO_M` scaling.

What `load_input.py` assumes:

- `InteriorDomain.shape[1]` is alongshore. If it exceeds `BarrierLength` the
  array is silently truncated; if it is smaller, `BarrierLength` is silently
  reduced to match. Either way the mismatch is never reported.
- `DuneStart` is sliced `[0:BarrierLength]`, so the dune array must be at
  least that long.
- Dune values are heights *above the berm*, not elevations
  (`newDuneHeight = newDuneElev - BermEl`), so they must be positive.

In [13]:
import yaml

WATER_CLAMP_DAM = -0.3   # SENTINEL_WATER_M / WATER_CLAMP_M from RUN_MANIFEST.txt
MAX_PLAUSIBLE_DAM = 2.0  # 20 m; a barrier island in metres would blow past this


def load_barrier3d_contract(parameter_path):
    """Reads the unit-relevant Barrier3D parameters from the CASCADE YAML.

    Mirrors the conversions in barrier3d/load_input.py so the values can be
    compared against the raw arrays on disk.

    Args:
        parameter_path: Path to the CASCADE parameter YAML.

    Returns:
        A dict with barrier_length_cells, mhw_dam and berm_el_dam.
    """
    with open(parameter_path, encoding="utf-8") as handle:
        params = yaml.safe_load(handle)

    mhw_dam = params["MHW"] / 10.0
    return {
        "barrier_length_cells": int(params["BarrierLength"] / 10.0),
        "mhw_dam": mhw_dam,
        "berm_el_dam": params["BermEl"] / 10.0 - mhw_dam,
    }


def check_domain_units(elevation_dam, dune_dam, contract):
    """Checks one domain's raw arrays against the Barrier3D input contract.

    Args:
        elevation_dam: Raw elevation array as stored on disk.
        dune_dam: Raw dune array as stored on disk.
        contract: Mapping from load_barrier3d_contract.

    Returns:
        A dict of check name -> bool, True when the array satisfies the check.
    """
    alongshore_cells = (elevation_dam.shape[1] if elevation_dam.ndim == 2
                        else None)
    return {
        "elevation is 2-D (cross_shore, alongshore)": elevation_dam.ndim == 2,
        "dune is 1-D (one crest per alongshore cell)": dune_dam.ndim == 1,
        "alongshore cells match dune length":
            alongshore_cells == dune_dam.size,
        f"alongshore >= BarrierLength ({contract['barrier_length_cells']})":
            alongshore_cells is not None
            and alongshore_cells >= contract["barrier_length_cells"],
        "dtype is floating point":
            np.issubdtype(elevation_dam.dtype, np.floating)
            and np.issubdtype(dune_dam.dtype, np.floating),
        "magnitudes are decameters, not metres":
            np.abs(elevation_dam).max() <= MAX_PLAUSIBLE_DAM,
        f"seaward floor at water clamp ({WATER_CLAMP_DAM} dam)":
            np.isclose(elevation_dam.min(), WATER_CLAMP_DAM),
        "dune heights positive (above berm)": dune_dam.min() > 0,
    }


BARRIER3D_CONTRACT = load_barrier3d_contract(HATTERAS_DATA_BASE / PARAMETER_FILE)

print(f"Contract from {PARAMETER_FILE}:")
print(f"  BarrierLength -> {BARRIER3D_CONTRACT['barrier_length_cells']} "
      f"alongshore cells")
print(f"  MHW           -> {BARRIER3D_CONTRACT['mhw_dam']:.3f} dam")
print(f"  BermEl        -> {BARRIER3D_CONTRACT['berm_el_dam']:.3f} dam "
      f"above MHW\n")

_results = {}
for _gis_id in range(HATTERAS_DOMAINS.first_gis_id,
                     HATTERAS_DOMAINS.last_gis_id + 1):
    _pad = HATTERAS_DOMAINS.gis_to_pad(_gis_id)
    _checks = check_domain_units(np.load(ELEVATION_FILE_PATHS[_pad]),
                                 np.load(DUNE_FILE_PATHS[_pad]),
                                 BARRIER3D_CONTRACT)
    for _name, _passed in _checks.items():
        _results.setdefault(_name, []).append((_gis_id, _passed))

print(f"{'Check':<46} {'domains passing':>15}")
_failed_any = False
for _name, _outcomes in _results.items():
    _failures = [gis_id for gis_id, passed in _outcomes if not passed]
    _failed_any = _failed_any or bool(_failures)
    print(f"  {_name:<44} {len(_outcomes) - len(_failures):>6}"
          f"/{len(_outcomes)}"
          + (f"   FAILED: {_failures[:8]}" if _failures else ""))

print("\n" + ("SOME CHECKS FAILED - do not run the model on these arrays"
               if _failed_any else
               "All units and shapes match what Barrier3D expects."))

Contract from Hatteras-CASCADE-parameters.yaml:
  BarrierLength -> 50 alongshore cells
  MHW           -> 0.036 dam
  BermEl        -> 0.134 dam above MHW

Check                                          domains passing
  elevation is 2-D (cross_shore, alongshore)       90/90
  dune is 1-D (one crest per alongshore cell)      90/90
  alongshore cells match dune length               90/90
  alongshore >= BarrierLength (50)                 90/90
  dtype is floating point                          90/90
  magnitudes are decameters, not metres            90/90
  seaward floor at water clamp (-0.3 dam)          90/90
  dune heights positive (above berm)               90/90

All units and shapes match what Barrier3D expects.


### 2.5 Browse the extractor's per-domain figures

2.4 samples five domains. This steps through all 90 using the figures
`HAT_dune_topo_extractor.py` wrote alongside the arrays, so a bad dune pick or
a mis-trimmed water edge is visible domain by domain.

Two sets are available, both under `<version>/figures/`:

- **`gis_vs_processed`** — three panels: raw GIS input (m NAVD88), the
  straightened picking frame with the dune search window and picked crest, and
  the processed CASCADE input actually written to disk, with the dune-height
  strip below. This is the one that shows what the extractor *did*.
- **`qc`** — the dune-detection QC panel for the same domain.

Drag the slider or focus it and use the arrow keys to cycle. The figure title
carries that domain's settings (search window, dunes found, mean dune height,
obliquity), which match the row for that domain in
`HAT_dune_topo_settings_<version>.csv`.

Note: `ipywidgets` is imported here rather than in Section 1 because it is an
optional dependency used only by this step — if it is missing, the rest of the
notebook still runs and you can call `show_domain_figure(gis_id)` directly.

In [14]:
FIGURES_DIR = DUNE_TOPO_DIR / "figures"
FIGURE_SETS = {
    "gis_vs_processed": "domain_{gis_id:03d}_gis_vs_processed.png",
    "qc": "domain_{gis_id:03d}_qc.png",
}
FIGURE_DISPLAY_WIDTH = 650  # rendered width; source PNGs are ~2060 px


def domain_figure_path(gis_id, figure_set):
    """Builds the path to one domain's extractor QC figure.

    Args:
        gis_id: GIS domain ID.
        figure_set: Key of FIGURE_SETS naming which figure set to use.

    Returns:
        A pathlib.Path to the PNG for that domain.

    Raises:
        KeyError: If figure_set is not a key of FIGURE_SETS.
        FileNotFoundError: If no figure exists for that domain.
    """
    filename = FIGURE_SETS[figure_set].format(gis_id=gis_id)
    path = FIGURES_DIR / figure_set / filename
    if not path.exists():
        raise FileNotFoundError(f"No {figure_set} figure for GIS {gis_id}: {path}")
    return path


def show_domain_figure(gis_id, figure_set="gis_vs_processed",
                       width=FIGURE_DISPLAY_WIDTH):
    """Displays one domain's extractor QC figure inline, downscaled to width.

    IPython.display.Image embeds local files and passes width only as output
    metadata, which some frontends ignore -- so the image is resampled to the
    requested width instead of being tagged with it.

    Args:
        gis_id: GIS domain ID.
        figure_set: Key of FIGURE_SETS naming which figure set to use.
        width: Display width in pixels.
    """
    path = domain_figure_path(gis_id, figure_set)
    with PIL.Image.open(path) as figure:
        height = round(figure.height * width / figure.width)
        preview = figure.convert("RGB").resize((width, height))

    print(f"GIS {gis_id}  ->  {path.name}  ({width}x{height} px)")
    display(preview)


for _figure_set in FIGURE_SETS:
    _missing = [gis_id
                for gis_id in range(HATTERAS_DOMAINS.first_gis_id,
                                    HATTERAS_DOMAINS.last_gis_id + 1)
                if not (FIGURES_DIR / _figure_set
                        / FIGURE_SETS[_figure_set].format(gis_id=gis_id)).exists()]
    print(f"{_figure_set:<18} "
          f"{HATTERAS_DOMAINS.num_real_domains - len(_missing)}"
          f"/{HATTERAS_DOMAINS.num_real_domains} figures"
          + (f"  MISSING: {_missing}" if _missing else ""))

try:
    import ipywidgets as widgets

    widgets.interact(
        show_domain_figure,
        gis_id=widgets.IntSlider(
            min=HATTERAS_DOMAINS.first_gis_id,
            max=HATTERAS_DOMAINS.last_gis_id,
            value=HATTERAS_DOMAINS.first_gis_id,
            step=1,
            description="GIS domain",
            continuous_update=False),
        figure_set=widgets.Dropdown(
            options=list(FIGURE_SETS),
            value="gis_vs_processed",
            description="figure set"),
        width=widgets.fixed(FIGURE_DISPLAY_WIDTH),
    )
except ImportError:
    print("\nipywidgets not installed -- call show_domain_figure(gis_id) "
          "directly, e.g. show_domain_figure(45)")

gis_vs_processed   90/90 figures
qc                 90/90 figures


interactive(children=(IntSlider(value=1, continuous_update=False, description='GIS domain', max=90, min=1), Dr…

## 3. Domain orientation — set `START_YEAR`, resolve period, shoreline_offset init
This one flip resolves SLR rate, storm file, island-offset file, road-setback file, and nourishment defaults -- worth showing all of it, not just the shoreline.

`SOURCE_SINK_PRESET` is also toggled here, but only as a string -- the actual `DOMAIN_BE_RATES_*` calibration dicts (213 lines, one entry per domain) are Section 8's job, not this section's. They only look period-coupled in the original script because both live in the same top-level dict there; that bundling isn't a real dependency of orientation itself.


In [ ]:
START_YEAR = 1984   # or 2004 -- the one thing you flip in this cell
SOURCE_SINK_PRESET = "calibrated"   # or "base" -- rates resolved in Section 8, not here

PERIOD_CONFIG = {
    1984: dict(
        end_year=2004,
        sea_level_rise_rate=0.004,  # m/yr - from duck_rslr_analysis.py
        storm_file=str(HATTERAS_DATA_BASE / "storms" / "hindcast_storms" / "1984_2004"
                       / "1984_2004_storms_v3_72.npy"),
        island_offset_file=str(HATTERAS_DATA_BASE / "2-brie-offset" / "hindcast_1984"
                               / f"Island_Dune_Offsets_1984_PADDED_{HATTERAS_DOMAINS.total_domains}.csv"),
        road_setback_file=str(HATTERAS_DATA_BASE / "roads" / "processed_offset" / "1984"
                              / "RoadSetback_1984.csv"),
        enable_nourishment=False,
        nourishment_volume=0,
    ),
    2004: dict(
        end_year=2024,
        sea_level_rise_rate=0.006,  # m/yr - from duck_rslr_analysis.py
        storm_file=str(HATTERAS_DATA_BASE / "storms" / "hindcast_storms" / "2004_2024"
                       / "2004_2024_storms_v3_72.npy"),
        island_offset_file=str(HATTERAS_DATA_BASE / "2-brie-offset" / "hindcast_2004"
                               / f"Island_Dune_Offsets_2004_PADDED_{HATTERAS_DOMAINS.total_domains}.csv"),
        road_setback_file=str(HATTERAS_DATA_BASE / "roads" / "processed_offset" / "2004"
                              / "RoadSetback_2004.csv"),  # UPDATE path once file is generated, if not already final
        enable_nourishment=True,   # historical BN injected per-year in time loop
        nourishment_volume=100,    # m^3/m default passed to Cascade init (threshold unused)
    ),
}

if START_YEAR not in PERIOD_CONFIG:
    raise ValueError(f"Invalid START_YEAR {START_YEAR}. Must be one of {list(PERIOD_CONFIG)}.")
if SOURCE_SINK_PRESET not in ("base", "calibrated"):
    raise ValueError(f"Invalid SOURCE_SINK_PRESET {SOURCE_SINK_PRESET!r}.")

_cfg = PERIOD_CONFIG[START_YEAR]
END_YEAR = _cfg["end_year"]
RUN_YEARS = END_YEAR - START_YEAR          # passed to CASCADE as time_step_count
SEA_LEVEL_RISE_RATE = _cfg["sea_level_rise_rate"]
STORM_FILE = _cfg["storm_file"]
ISLAND_OFFSET_FILE = _cfg["island_offset_file"]
ROAD_SETBACK_FILE = _cfg["road_setback_file"]
ENABLE_NOURISHMENT = _cfg["enable_nourishment"]
NOURISHMENT_VOLUME = _cfg["nourishment_volume"]
RUN_NAME_SUFFIX = "full_calibrated"   # <- edit this to label each experiment
RUN_NAME_BASE = f"HAT_{START_YEAR}_{END_YEAR}_{RUN_NAME_SUFFIX}"

print("=" * 80)
print("HATTERAS ISLAND CASCADE - PERIOD CONFIGURATION")
print("=" * 80)
print(f"Period:               {START_YEAR}-{END_YEAR}  ({RUN_YEARS} years)")
print(f"SLR rate:             {SEA_LEVEL_RISE_RATE * 1000:.1f} mm/yr")
print(f"SOURCE_SINK_PRESET:   '{SOURCE_SINK_PRESET}'  (rates resolved in Section 8)")
print(f"Storm file:           {Path(STORM_FILE).name}")
print(f"Island offset file:   {Path(ISLAND_OFFSET_FILE).name}")
print(f"Road setback file:    {Path(ROAD_SETBACK_FILE).name}")
print(f"Nourishment enabled:  {ENABLE_NOURISHMENT}  (default volume: {NOURISHMENT_VOLUME} m^3/m)")
print("=" * 80)

if not Path(ROAD_SETBACK_FILE).is_file():
    print(f"WARNING: ROAD_SETBACK_FILE not found (Section 5 will need it): {ROAD_SETBACK_FILE}")

# --- QC PLOT: island offset by GIS domain, BOTH periods overlaid --------------
# so a bad START_YEAR flip is obvious immediately -- not just the active one.
gis_ids = list(range(HATTERAS_DOMAINS.first_gis_id, HATTERAS_DOMAINS.last_gis_id + 1))

fig, ax = plt.subplots(figsize=(12, 4.5))
island_offset_by_period = {}

for _year, _cfg_y in PERIOD_CONFIG.items():
    _path = Path(_cfg_y["island_offset_file"])
    if not _path.is_file():
        print(f"  WARNING: island offset file for {_year} not found: {_path}")
        continue
    _raw = np.loadtxt(_path, skiprows=1, delimiter=",")
    _dam = _raw / 10.0  # m -> dam
    island_offset_by_period[_year] = _dam
    _real = _dam[HATTERAS_DOMAINS.start_real_index:HATTERAS_DOMAINS.end_real_index]
    _active = (_year == START_YEAR)
    ax.plot(gis_ids, _real, label=f"{_year} (active)" if _active else f"{_year}",
            lw=2.4 if _active else 1.3, alpha=1.0 if _active else 0.55,
            zorder=3 if _active else 2)

# active period's offset, needed later for Cascade(shoreline_offset=...) in Section 12
island_offset_dam = island_offset_by_period[START_YEAR]
print(f"  island_offset_dam (active, {START_YEAR}): {island_offset_dam.size} domains (dam)")

ax.set_xlabel(f"GIS Domain ID ({HATTERAS_DOMAINS.first_gis_id}-{HATTERAS_DOMAINS.last_gis_id})")
ax.set_ylabel("Island/dune offset (dam)")
ax.set_title(f"Section 3 QC: island offset by domain, both periods -- active = {START_YEAR}")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## 4. Forcings — SLR, storms, and source/sink (background erosion)
All three are pure `PERIOD_CONFIG`/`START_YEAR` lookups from Section 3, with no other section as a prerequisite -- grouped here as "the period-dependent rate inputs to the run," split into 4.1/4.2/4.3 the same way Section 2 was.


### 4.1 Sea level rise rate


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
_years = list(PERIOD_CONFIG.keys())
_rates_mm = [PERIOD_CONFIG[y]["sea_level_rise_rate"] * 1000 for y in _years]
_colors = ["#FF8C00" if y == START_YEAR else "0.7" for y in _years]
ax.bar([str(y) for y in _years], _rates_mm, color=_colors)
ax.set_ylabel("SLR rate (mm/yr)")
ax.set_title(f"Section 4.1 QC: SLR rate by period -- active = {START_YEAR}")
for i, v in enumerate(_rates_mm):
    ax.text(i, v + 0.05, f"{v:.1f}", ha="center", fontsize=9)
fig.tight_layout()
plt.show()


### 4.2 Storms


In [ ]:
# The storm catalog's internal array structure (Hs/duration/count per storm) is
# defined by HAT_create_storms.py, not parsed here -- this only confirms the
# active file exists and reports what np.load actually sees, rather than
# guessing at a structure this notebook doesn't otherwise use.
storm_path = Path(STORM_FILE)
if not storm_path.is_file():
    print(f"WARNING: STORM_FILE not found: {storm_path}")
else:
    _storms = np.load(storm_path, allow_pickle=True)
    print(f"STORM_FILE: {storm_path.name}")
    print(f"  shape={_storms.shape}  dtype={_storms.dtype}")
    if _storms.dtype == object or _storms.ndim == 0:
        print("  (structured/object array -- inspect with HAT_create_storms.py's "
              "own loader for the real field layout, not shape alone)")


### 4.3 Source/sink (background erosion rate)
`DOMAIN_BE_RATES_BASE_BY_PERIOD` / `DOMAIN_BE_RATES_CALIBRATED_BY_PERIOD` copied verbatim from the original script (lines 251-456) -- real per-domain calibration data, not retyped. `SOURCE_SINK_PRESET` was set as a string back in Section 3; this is where it's actually resolved into `DOMAIN_BE_RATES`, now that START_YEAR is known.


In [ ]:
# --- Verbatim from original script, lines 251-456 ------------------------------

DOMAIN_BE_RATES_BASE_BY_PERIOD = {
    1984: {
        1:  -40, #-40
        90:  15, # 15
    },
    2004: {
        1:   35, #35
        90:  35, #35
    },
}

# "Calibrated" preset
DOMAIN_BE_RATES_CALIBRATED_BY_PERIOD = {
    1984: {
          1: -40.0,  # LOCKED — use your solved value, not 0.0
          2: -0.6,  # Cape Point / Shoal Dynamics
          3: -0.7,  # Cape Point / Shoal Dynamics
          4: 0.0,
          5: +1.2,  # Cape Point / Shoal Dynamics
          6: +2.0,  # Cape Point / Shoal Dynamics
          7: +0.9,  # Cape Point / Shoal Dynamics
          8: +1.1,  # Cape Point / Shoal Dynamics
          9: 0.0,
         10: -0.9,  # Cape Point / Shoal Dynamics
         11: -1.3,  # Buxton–Avon Transition
         12: -0.9,  # Buxton–Avon Transition
         13: -0.6,  # Buxton–Avon Transition
         14: 0.0,
         15: 0.0,
         16: 0.0,
         17: +0.6,  # Buxton–Avon Transition
         18: +0.6,  # Buxton–Avon Transition
         19: +0.6,  # Buxton–Avon Transition
         20: 0.0,
         21: 0.0,
         22: 0.0,
         23: 0.0,
         24: +0.3,  # Avon
         25: +0.3,  # Avon
         26: +0.8,  # Avon
         27: +1.4,  # Avon
         28: +1.9,  # Avon
         29: +2.3,  # Avon
         30: +2.2,  # Avon
         31: +2.2,  # Avon
         32: +1.9,  # Mid-island
         33: +1.4,  # Mid-island
         34: +1.0,  # Mid-island
         35: 0.0,
         36: 0.0,
         37: 0.0,
         38: 0.0,
         39: 0.0,
         40: 0.0,
         41: 0.0,
         42: 0.0,
         43: 0.0,
         44: +0.4,  # Mid-island
         45: 0.0,
         46: 0.0,
         47: 0.0,
         48: -0.3,  # Mid-island
         49: -1.0,  # Mid-island
         50: -1.0,  # Mid-island
         51: -1.2,  # Mid-island
         52: -1.3,  # Mid-island
         53: -1.3,  # Mid-island
         54: -1.2,  # Mid-island
         55: -1.0,  # Mid-island
         56: -1.0,  # Mid-island
         57: -0.3,  # Mid-island
         58: 0.0,
         59: 0.0,
         60: 0.0,
         61: 0.0,
         62: +0.4,  # Wimble Shoals Influence
         63: 0.0,
         64: 0.0,
         65: 0.0,
         66: 0.0,
         67: 0.0,
         68: +0.7,  # Wimble Shoals Influence
         69: +1.2,  # Wimble Shoals Influence
         70: +1.6,  # Wimble Shoals Influence
         71: +1.9,  # Wimble Shoals Influence
         72: +2.2,  # Wimble Shoals Influence
         73: +2.4,  # Wimble Shoals Influence
         74: +2.6,  # Wimble Shoals Influence
         75: +2.5,  # Tri-Village / Rodanthe
         76: +1.7,  # Tri-Village / Rodanthe
         77: +1.3,  # Tri-Village / Rodanthe
         78: +0.7,  # Tri-Village / Rodanthe
         79: 0.0,
         80: -1.3,  # Tri-Village / Rodanthe
         81: -1.9,  # Tri-Village / Rodanthe
         82: -2.1,  # Tri-Village / Rodanthe
         83: -2.3,  # Tri-Village / Rodanthe
         84: -2.5,  # Pea Island NWR
         85: -2.5,  # Pea Island NWR
         86: -2.4,  # Pea Island NWR
         87: -2.0,  # Pea Island NWR
         88: -1.6,  # Pea Island NWR
         89: -1.0,  # Pea Island NWR
         90: 15.0,  # LOCKED — use your solved value, not 0.0
    },

    2004: {
          1: 35.0,  # LOCKED — use your solved value, not 0.0
          2: +1.0,  # Cape Point / Shoal Dynamics
          3: +1.6,  # Cape Point / Shoal Dynamics
          4: +1.6,  # Cape Point / Shoal Dynamics
          5: +1.2,  # Cape Point / Shoal Dynamics
          6: -1.5,  # Cape Point / Shoal Dynamics
          7: 0.0,
          8: +1.1,  # Cape Point / Shoal Dynamics
          9: +2.2,  # Cape Point / Shoal Dynamics
         10: +3.1,  # Cape Point / Shoal Dynamics
         11: +3.4,  # Buxton–Avon Transition
         12: +3.4,  # Buxton–Avon Transition
         13: +3.3,  # Buxton–Avon Transition
         14: +3.2,  # Buxton–Avon Transition
         15: +3.0,  # Buxton–Avon Transition
         16: +3.1,  # Buxton–Avon Transition
         17: +3.0,  # Buxton–Avon Transition
         18: +2.9,  # Buxton–Avon Transition
         19: +2.6,  # Buxton–Avon Transition
         20: +2.2,  # Buxton–Avon Transition
         21: +1.9,  # Avon
         22: +1.4,  # Avon
         23: +0.8,  # Avon
         24: +0.3,  # Avon
         25: +0.3,  # Avon
         26: +0.8,  # Avon
         27: +1.4,  # Avon
         28: +1.9,  # Avon
         29: +2.3,  # Avon
         30: +3.1,  # Avon
         31: +3.6,  # Avon
         32: +3.8,  # Mid-island
         33: +3.7,  # Mid-island
         34: +3.6,  # Mid-island
         35: +3.5,  # Mid-island
         36: +3.3,  # Mid-island
         37: +3.1,  # Mid-island
         38: +2.7,  # Mid-island
         39: +2.4,  # Mid-island
         40: +2.1,  # Mid-island
         41: +1.8,  # Mid-island
         42: +1.5,  # Mid-island
         43: +1.1,  # Mid-island
         44: +0.4,  # Mid-island
         45: 0.0,
         46: 0.0,
         47: 0.0,
         48: -0.3,  # Mid-island
         49: 0.0,
         50: -1.0,  # Mid-island
         51: -1.2,  # Mid-island
         52: -1.3,  # Mid-island
         53: -1.3,  # Mid-island
         54: -1.2,  # Mid-island
         55: -1.0,  # Mid-island
         56: 0.0,
         57: -0.3,  # Mid-island
         58: 0.0,
         59: 0.0,
         60: 0.0,
         61: 0.0,
         62: +0.4,  # Wimble Shoals Influence
         63: +1.1,  # Wimble Shoals Influence
         64: +1.6,  # Wimble Shoals Influence
         65: +2.0,  # Wimble Shoals Influence
         66: +2.4,  # Wimble Shoals Influence
         67: +2.8,  # Wimble Shoals Influence
         68: +3.3,  # Wimble Shoals Influence
         69: +3.7,  # Wimble Shoals Influence
         70: +4.0,  # Wimble Shoals Influence
         71: +4.2,  # Wimble Shoals Influence
         72: +4.1,  # Wimble Shoals Influence
         73: +4.0,  # Wimble Shoals Influence
         74: +3.7,  # Wimble Shoals Influence
         75: +3.4,  # Tri-Village / Rodanthe
         76: +3.0,  # Tri-Village / Rodanthe
         77: +2.7,  # Tri-Village / Rodanthe
         78: +2.4,  # Tri-Village / Rodanthe
         79: +2.3,  # Tri-Village / Rodanthe
         80: +2.0,  # Tri-Village / Rodanthe
         81: +1.7,  # Tri-Village / Rodanthe
         82: +1.4,  # Tri-Village / Rodanthe
         83: +1.0,  # Tri-Village / Rodanthe
         84: +0.7,  # Pea Island NWR
         85: 0.0,
         86: +0.5,  # Pea Island NWR
         87: +0.6,  # Pea Island NWR
         88: +0.7,  # Pea Island NWR
         89: +1.2,  # Pea Island NWR
         90: 35.0,  # LOCKED — use your solved value, not 0.0
    },  #
}


if DOMAIN_BE_RATES_CALIBRATED_BY_PERIOD.get(2004) is None:
    DOMAIN_BE_RATES_CALIBRATED_BY_PERIOD[2004] = dict(DOMAIN_BE_RATES_CALIBRATED_BY_PERIOD[1984])
    DOMAIN_BE_RATES_CALIBRATED_2004_IS_PLACEHOLDER = True
else:
    DOMAIN_BE_RATES_CALIBRATED_2004_IS_PLACEHOLDER = False

# --- Resolve DOMAIN_BE_RATES for the active period + preset --------------------
_BE_PRESETS_THIS_PERIOD = {
    "base": DOMAIN_BE_RATES_BASE_BY_PERIOD[START_YEAR],
    "calibrated": DOMAIN_BE_RATES_CALIBRATED_BY_PERIOD[START_YEAR],
}
if SOURCE_SINK_PRESET not in _BE_PRESETS_THIS_PERIOD:
    raise ValueError(f"SOURCE_SINK_PRESET {SOURCE_SINK_PRESET!r} not valid; "
                     f"must be one of {list(_BE_PRESETS_THIS_PERIOD)}.")
DOMAIN_BE_RATES = _BE_PRESETS_THIS_PERIOD[SOURCE_SINK_PRESET]

# Loud, impossible-to-miss warning -- catches the exact mistake that caused
# "editing the 2004 calibrated values doesn't appear to do anything."
if (START_YEAR == 2004 and SOURCE_SINK_PRESET == "calibrated"
        and DOMAIN_BE_RATES_CALIBRATED_2004_IS_PLACEHOLDER):
    print("=" * 80)
    print("WARNING: START_YEAR=2004, SOURCE_SINK_PRESET='calibrated' is still")
    print("         using the 1984 PLACEHOLDER interior shape.")
    print("=" * 80)

print(f"SOURCE_SINK_PRESET: '{SOURCE_SINK_PRESET}'  "
      f"({len(DOMAIN_BE_RATES)} non-default domain(s) for {START_YEAR})")

# --- QC PLOT: base vs calibrated BE rate by domain, both periods ---------------
def _be_rates_array(sparse_dict, domains):
    """Expands a {gis_id: rate} sparse dict to a full per-real-domain array (default 0)."""
    arr = np.zeros(domains.num_real_domains)
    for gis_id, rate in sparse_dict.items():
        arr[gis_id - domains.first_gis_id] = rate
    return arr


gis_ids = list(range(HATTERAS_DOMAINS.first_gis_id, HATTERAS_DOMAINS.last_gis_id + 1))
fig, (ax_base, ax_cal) = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)

for _year in PERIOD_CONFIG:
    _active = (_year == START_YEAR)
    _base_arr = _be_rates_array(DOMAIN_BE_RATES_BASE_BY_PERIOD[_year], HATTERAS_DOMAINS)
    _cal_arr = _be_rates_array(DOMAIN_BE_RATES_CALIBRATED_BY_PERIOD[_year], HATTERAS_DOMAINS)
    _lbl = f"{_year}" + (" (active)" if _active else "")
    _lw, _alpha = (2.2, 1.0) if _active else (1.2, 0.5)
    ax_base.plot(gis_ids, _base_arr, label=_lbl, lw=_lw, alpha=_alpha, marker="o", markersize=3)
    ax_cal.plot(gis_ids, _cal_arr, label=_lbl, lw=_lw, alpha=_alpha)

ax_base.set_title("'base' preset")
ax_cal.set_title("'calibrated' preset")
for _ax in (ax_base, ax_cal):
    _ax.axhline(0, color="gray", lw=0.8, ls="--")
    _ax.set_xlabel(f"GIS Domain ID ({HATTERAS_DOMAINS.first_gis_id}-{HATTERAS_DOMAINS.last_gis_id})")
    _ax.grid(alpha=0.3)
    _ax.legend(fontsize=8)
ax_base.set_ylabel("Background erosion rate (m/yr)")
fig.suptitle(f"Section 4.3 QC: BE rates by domain, both periods -- "
             f"active preset = '{SOURCE_SINK_PRESET}' ({START_YEAR})")
fig.tight_layout()
plt.show()


## 5. `roadway_manager` — setbacks + historical events + road-exists-AND-not-community rule


In [ ]:
# load road_setbacks_full from ROAD_SETBACK_FILE, padded to TOTAL_DOMAINS -- orig. lines 1077-1089

# HISTORICAL_ROAD_EVENTS -- orig. lines 951-996 (1989 relocation, 1999 relocation, 2022 bridge)
# ENABLE_HISTORICAL_ROAD_RELOCATIONS = False   # currently off

# ROADWAY_MANAGEMENT_ON: road span (GIS 9-90) minus permanent community zones only
# -- from cascade_pipeline.community_zones import COMMUNITY_ZONES, build_roadway_management_on
# (Buxton 7-8, Avon 21-31, Tri-Village 68-83 -- NOT the wider nourishment footprint,
#  confirmed against the reference table)

# QC PLOT: ROADWAY_MANAGEMENT_ON as a bar/strip by GIS domain, with markers at
# the historical event years/domains so the exclusions and events are both visible at once


## 6. `beach_dune_manager` — nourishment schedule + overwash filter
Two things bundled in one CASCADE module: always-on overwash filtering/dune fixing for community zones, and the separate calendar-year nourishment schedule.


In [ ]:
# HAT_BN_YEARS, HAT_BN_VOLUME_BY_DOMAIN -- orig. lines 1006-1057
# (Rodanthe D85-88 2014, Buxton D6-15 2022, Avon D23-26 2022)

# build_nourishment_arrays_from_manual_inputs() -- orig. lines 1156-1217
# -> HIST_NOURISH_ON, HIST_NOURISH_VOL (per-year dicts, used later in the time loop)

# OVERWASH_FILTER + NOURISHMENT_MANAGEMENT_ON (renamed beach_nourishment_module downstream):
# from cascade_pipeline.community_zones import build_overwash_filter, build_beach_nourishment_module
# beach_nourishment_module = True for COMMUNITY_ZONES UNION HAT_BN_VOLUME_BY_DOMAIN.keys()
# (community zones get a real filter %; nourishment-only domains get default/0%,
#  they're inter-village/refuge, not developed communities -- confirmed against reference table)

# QC PLOT: OVERWASH_FILTER by GIS domain, and beach_nourishment_module as a bar/strip,
# with the historic nourishment years/domains marked


## 7. `hard_structures` / groin — Buxton groin config
New QC plot that doesn't exist in the old script: deterioration_fraction(t) for both periods, since Period 1 exercises the ramp mid-run and Period 2 starts at the floor.


In [ ]:
# GROIN_ENABLED, GROIN_UPDRIFT_GIS=6, GROIN_DOWNDRIFT_GIS=5, GROIN_INSTALL_YEAR=1969,
# GROIN_TRAPPING_RATE_M_YR, GROIN_DETERIORATION_* -- orig. lines 233-252

# smoke-test the deferred import: can HAT_groin_module.GroinCallback actually be imported?
# (orig. does this inside run_cascade_simulation, lines 2126-2132 -- worth doing here instead,
#  fails loudly now rather than mid-run)

# QC PLOT (new): deterioration_fraction(t) for START_YEAR=1984 and START_YEAR=2004 side by side
# -- confirms the ramp lands where expected (Period 1: mid-run years 12-19; Period 2: floor from year 0)

# NOTE: can't verify the actual pre-AST hook placement without cascade_groin.py (not yet reviewed) --
# upload it if you want that specific piece checked against cascade.py's update() order


## 8. CoastSat target rates — LOESS windows


In [ ]:
# NOW LIVES IN cascade_pipeline.coastsat_loess (extracted this session) -- this cell just
# configures datasets and calls the package, nothing inline anymore.
#
# COASTSAT_DATASETS -> list of CoastSatDataset(label=..., period_start=..., csv_path=...)
#   (domain_col/rate_col/transect_id_col default to the standard column names;
#   only pass them if your CSVs differ)
# loess_config = LoessConfig(window_domains=(7, 10), skip_southern_domains=10)
#   -- 10-domain window is the calibration reference; matches DEFAULT_LOESS
# cs_series = build_coastsat_series(COASTSAT_DATASETS, active_period_start=START_YEAR, loess_config=loess_config)
#   -- replaces the load+LOESS loop that used to live in main(); tags each
#   dataset active/reference automatically by comparing period_start to START_YEAR

# QC PLOT: raw transect LRR scatter + LOESS lines at each window width, for the active period
# (load_transect_data(ds) on its own, per dataset, if you want to inspect one before smoothing)


## 9. Plotting functions
Extracted to `cascade_pipeline.plotting` (this session) -- `shoreline_gif.py` and `rate_comparison.py`, plus shared `cascade_pipeline.annotations` for the geographic layer both use. The ~20 module-level constants (GIF toggles, annotation config, domain/period config) became explicit dataclass configs (`GifConfig`, `AnnotationConfig`, `RateComparisonConfig`) and a `RunInfo` bundling run_name/run_dir/start_year/end_year/Hs/sign-convention, passed in rather than read off globals -- that's what makes it importable here independent of the run script's state. Function names are unchanged (`make_shoreline_gif`, `make_all_shoreline_gifs`, `add_geographic_annotations`, etc.) so this is a drop-in for any other script that already calls them.

In [ ]:
# get_x_s_TS / build_shoreline_matrix -> cascade_pipeline.shoreline (also has
# compute_change_rate, new -- replaces the change_rate calc that used to be inline in main())

# GIF config toggles -> GifConfig(fps=3, ocean_at_bottom=True, ...); GIF_JOBS stays a
# plain list of dicts (same shape as before), passed to make_all_shoreline_gifs

# gif_config = GifConfig()
# GIF_JOBS = [
#     dict(range="real",  mode="displacement"),
#     dict(range="real",  mode="position", auto_open=True),
#     dict(range="groin", mode="position",   pad=9),
#     dict(range="groin", mode="difference", pad=9),
# ]

# make_shoreline_gif / make_all_shoreline_gifs + private helpers
# (_open_file, _slug, _groin_zoom_window, _select_groins, _resolve_gif_domain_range)
# -> cascade_pipeline.plotting.shoreline_gif, unchanged public names


## 10. Define `run_cascade_simulation()`


In [ ]:
# adapted from orig. lines 2052-2309, with one fix:
#
# in the "bridge" historical event handler, after terminated_road_pads.add(pad):
#   new_flags = list(cascade.roadway_management_module)
#   for pad in pad_indices: new_flags[pad] = False
#   cascade.roadway_management_module = new_flags
# (previously tracked in a local set that nothing ever read -- GIS 82-88 kept full road
# management after the 2022 bridge event instead of switching off as documented)
#
# everything else the same: Cascade(...) init, groin callback attach, per-year loop with
# historical nourishment + road event handling, cascade.update(), save()


## 11. Initialize Cascade — single config, no sweep


In [ ]:
# RMIN, RMAX, DUNE_DESIGN_ELEVATION, DUNE_MINIMUM_ELEVATION, ROAD_ELEVATION, ROAD_WIDTH
# -- orig. lines 2317-2323, unchanged

# one Hs value, not WAVE_HEIGHTS_TO_TEST -- no sweep loop here, that's a separate script
# Hs = 1.0
# run_name = f'HAT_{START_YEAR}_{END_YEAR}_{RUN_NAME_SUFFIX}'


## 12. Run the loop, then generate the GIF


In [ ]:
# cascade = run_cascade_simulation(nt=RUN_YEARS, name=run_name, ...)
# no per-step plotting inside the loop -- confirmed not needed

# after the run finishes:
# shoreline_m = build_shoreline_matrix(cascade)
# run = RunInfo(run_name=run_name, run_dir=run_dir, start_year=START_YEAR, end_year=END_YEAR,
#               Hs=Hs, flip_sign_model=FLIP_SIGN_MODEL, background_erosion_on=USE_BACKGROUND_EROSION)
# make_all_shoreline_gifs(shoreline_m, run, GIF_JOBS, baseline_npy=GIF_BASELINE_NPY, gif_config=gif_config)

# rate/annotated figures (cascade_pipeline.plotting.rate_comparison) also take `run` --
# change_rate = compute_change_rate(shoreline_m, span_years=RUN_YEARS, flip_sign=FLIP_SIGN_MODEL)
# plot_rate_comparison(change_rate, cs_series, run, real_domains_only=PLOT_REAL_DOMAINS_ONLY, ...)
# plot_annotated_rate_comparison(change_rate, cs_series, run, ...)
